# nanoPeriGPT — Incremental Experiments

Peridynamic attention + DEER layer-parallelism for transformers.

Each cell runs one experiment. Compare val_loss between cells to track progress.

In [ ]:
# Setup: upload project files or clone from repo
import os
import torch

# If running from uploaded zip:
# !unzip -q perigpt.zip -d perigpt
# os.chdir('perigpt')

# If cloning from git:
# !git clone <your-repo-url> perigpt
# os.chdir('perigpt')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Prepare data
!python data/shakespeare_char/prepare.py

In [ ]:
# Phase 1: Vanilla nanoGPT Baseline
!python train.py config/baseline.py

In [ ]:
# Phase 2: Bond-Based Peridynamic Attention (horizon=32)
!python train.py config/baseline.py \
    --out_dir=out-peri-h32 \
    --wandb_run_name=peri_h32 \
    --experiment_name=peri_h32 \
    --attention_type=peridynamic \
    --horizon=32

In [ ]:
# Phase 3: Horizon Sweep
for h in [16, 64, 128]:
    print(f'\n{"="*60}')
    print(f'Horizon = {h}')
    print(f'{"="*60}')
    !python train.py config/baseline.py \
        --out_dir=out-peri-h{h} \
        --wandb_run_name=peri_h{h} \
        --experiment_name=peri_h{h} \
        --attention_type=peridynamic \
        --horizon={h}

In [ ]:
# Check results so far
import pandas as pd
df = pd.read_csv('results.tsv', sep='\t')
print(df.sort_values('val_loss').to_string(index=False))

In [ ]:
# Phase 5: Block Attention Residual (standard attn + block_attn depth)
!python train.py config/baseline.py \
    --out_dir=out-blockres \
    --wandb_run_name=block_attn_res \
    --experiment_name=block_attn_res \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Phase 6: Combined — Peri Attention + Block Attn Residual
# Use best horizon from Phase 3
BEST_HORIZON = 32  # <-- update based on Phase 3 results
!python train.py config/baseline.py \
    --out_dir=out-peri-blockres \
    --wandb_run_name=peri_blockres \
    --experiment_name=peri_blockres \
    --attention_type=peridynamic \
    --horizon={BEST_HORIZON} \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Phase 11: Hybrid Attention (Peri local + sparse global)
!python train.py config/baseline.py \
    --out_dir=out-hybrid \
    --wandb_run_name=hybrid_a8 \
    --experiment_name=hybrid_a8 \
    --attention_type=hybrid \
    --horizon={BEST_HORIZON} \
    --n_global_anchors=8

In [ ]:
# Final results comparison
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('results.tsv', sep='\t')
df = df.sort_values('val_loss')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if s == 'KEEP' else 'red' for s in df['status']]
ax.barh(df['experiment'], df['val_loss'], color=colors)
ax.set_xlabel('Validation Loss')
ax.set_title('nanoPeriGPT Experiments — shakespeare_char')
ax.invert_yaxis()
for i, (v, t) in enumerate(zip(df['val_loss'], df['train_loss'])):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('progress.png', dpi=150)
plt.show()
print(df.to_string(index=False))

In [ ]:
# Sample from best model
BEST_DIR = 'out-baseline'  # <-- update to best experiment's out_dir
!python sample.py --out_dir={BEST_DIR} --num_samples=3 --max_new_tokens=500

# Mixed-Domain Experiments

The critical test: does peridynamic damage learn to break bonds at domain boundaries?

Dataset: Shakespeare + Python code + JSON + Federalist Papers interleaved in random chunks (64-384 chars). Domain boundaries fall within the 256-char context window.

In [ ]:
# Prepare mixed-domain data
!python data/mixed_domain/prepare.py

In [ ]:
# Mixed-domain: Baseline (standard attention)
!python train.py config/mixed_baseline.py

In [ ]:
# Mixed-domain: Peridynamic attention (bond-based) — horizon sweep
for h in [32, 64, 128]:
    print(f'\n{"="*60}')
    print(f'Mixed-domain Peridynamic h={h}')
    print(f'{"="*60}')
    !python train.py config/mixed_baseline.py \
        --out_dir=out-mixed-peri-h{h} \
        --experiment_name=mixed_peri_h{h} \
        --attention_type=peridynamic \
        --horizon={h}

In [ ]:
# Mixed-domain: Block attention residual
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-blockres \
    --experiment_name=mixed_block_attn_res \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Mixed-domain: Peri + Block Attn Res combined
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-peri-blockres \
    --experiment_name=mixed_peri_blockres \
    --attention_type=peridynamic \
    --horizon=64 \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Mixed-domain: Hybrid attention
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-hybrid \
    --experiment_name=mixed_hybrid_a8 \
    --attention_type=hybrid \
    --horizon=64 \
    --n_global_anchors=8

In [ ]:
# Mixed-domain: State-based peridynamic attention (THE BIG ONE)
for h in [64, 128]:
    !python train.py config/mixed_baseline.py \
        --out_dir=out-mixed-state-h{h} \
        --experiment_name=mixed_state_peri_h{h} \
        --attention_type=state_peridynamic \
        --horizon={h}

In [ ]:
# Compare ALL results — shakespeare vs mixed-domain
import pandas as pd

df = pd.read_csv('results.tsv', sep='\t')
df = df.sort_values('val_loss')

print("=" * 80)
print("ALL EXPERIMENTS")
print("=" * 80)
print(df.to_string(index=False))

# Separate shakespeare vs mixed results
print("\n" + "=" * 80)
print("SHAKESPEARE (homogeneous)")
print("=" * 80)
shk = df[~df['experiment'].str.startswith('mixed')]
print(shk.to_string(index=False))

print("\n" + "=" * 80)
print("MIXED-DOMAIN (heterogeneous)")
print("=" * 80)
mix = df[df['experiment'].str.startswith('mixed')]
print(mix.to_string(index=False))

# The key question: does peri attention rank differently on mixed vs shakespeare?
print("\n" + "=" * 80)
print("KEY COMPARISON: Peri vs Baseline ranking change")
print("=" * 80)
for dataset_name, sub in [("Shakespeare", shk), ("Mixed", mix)]:
    if len(sub) < 2:
        continue
    baseline = sub[sub['experiment'].str.contains('baseline')]['val_loss'].values
    peri = sub[sub['experiment'].str.contains('peri_h')]['val_loss'].values
    if len(baseline) > 0 and len(peri) > 0:
        best_peri = min(peri)
        b = baseline[0]
        diff = best_peri - b
        direction = "WORSE" if diff > 0 else "BETTER"
        print(f"  {dataset_name}: baseline={b:.4f}, best_peri={best_peri:.4f}, "
              f"delta={diff:+.4f} ({direction})")

In [ ]:
# Mixed-domain: Increased bond_dim (bond_dim_ratio=2 means bd=32 instead of 16)
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-peri-bd2 \
    --experiment_name=mixed_peri_bd2 \
    --attention_type=peridynamic \
    --horizon=128 \
    --bond_dim_ratio=2

In [ ]:
# Mixed-domain: Block attn-res WITH damage gating (fixes cross-domain contamination)
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-blockdmg \
    --experiment_name=mixed_blockres_damage \
    --residual_type=block_attn \
    --depth_block_size=3 \
    --block_damage=True

In [ ]:
# Mixed-domain: Full stack — state-based peri + damage-gated block_attn_res
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-full \
    --experiment_name=mixed_full_stack \
    --attention_type=state_peridynamic \
    --horizon=128 \
    --residual_type=block_attn \
    --depth_block_size=3 \
    --block_damage=True

# Damage Analysis

The key scientific question: does damage correlate with domain boundaries?

In [ ]:
# Run damage analysis on the best peri model
# Update out_dir to match your best mixed-domain peri checkpoint
!python analyze_damage.py --out_dir=out-mixed-peri-h128 --n_samples=100